# 👥 Week 2 — Customer Segmentation (RFM + K-Means)
**Smart E-Commerce Analytics Platform**

**Dataset:** Olist Brazilian E-Commerce → `orders_clean.csv` (96,477 delivered orders)

**This notebook covers:**
1. Load cleaned Olist orders
2. Compute RFM (Recency, Frequency, Monetary) per customer
3. Log-transform & standardize RFM features
4. K-Means clustering (Elbow method to find optimal k)
5. Segment labeling & profiling
6. PCA visualization of clusters

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
print('Libraries loaded ✅')

## 1. Load Cleaned Olist Orders

In [ ]:
df = pd.read_csv('../data/orders_clean.csv', parse_dates=['transaction_date'])
print(f'Shape: {df.shape}')
print(f'Date range: {df["transaction_date"].min().date()} → {df["transaction_date"].max().date()}')
print(f'Unique customers: {df["customer_id"].nunique():,}')
display(df.head(3))

## 2. Compute RFM Features

In [ ]:
# Snapshot date = day after last transaction
snapshot = df['transaction_date'].max() + pd.Timedelta(days=1)
print(f'Snapshot date: {snapshot.date()}')

rfm = df.groupby('customer_id').agg(
    Recency   = ('transaction_date', lambda x: (snapshot - x.max()).days),
    Frequency = ('order_id',         'count'),
    Monetary  = ('total_amount',     'sum'),
).reset_index()
rfm['Monetary'] = rfm['Monetary'].round(2)

print(f'\nRFM shape: {rfm.shape}')
display(rfm.describe().round(2))

In [ ]:
# Visualize raw RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, color in zip(axes, ['Recency','Frequency','Monetary'],
                           ['steelblue','orange','green']):
    ax.hist(rfm[col].clip(upper=rfm[col].quantile(0.99)), bins=40,
            color=color, edgecolor='white', alpha=0.8)
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
plt.suptitle('Raw RFM Distributions (right-skewed → need log transform)', y=1.02)
plt.tight_layout()
plt.savefig('../report/rfm_raw_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Log-Transform & Standardize

In [ ]:
# Log1p transform to reduce skewness
rfm_log = rfm[['Recency','Frequency','Monetary']].copy()
rfm_log['Recency']   = np.log1p(rfm_log['Recency'])
rfm_log['Frequency'] = np.log1p(rfm_log['Frequency'])
rfm_log['Monetary']  = np.log1p(rfm_log['Monetary'])

# Standardize
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

print('After log-transform + standardization:')
print(pd.DataFrame(rfm_scaled, columns=['Recency','Frequency','Monetary']).describe().round(3))

# Visualize after transform
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, i in zip(axes, ['Recency','Frequency','Monetary'], range(3)):
    ax.hist(rfm_scaled[:, i], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'{col} (log-scaled)')
plt.suptitle('RFM After Log-Transform & Standardization', y=1.02)
plt.tight_layout()
plt.show()

## 4. Elbow Method — Find Optimal K

In [ ]:
inertias    = []
sil_scores  = []
K_range     = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, labels))
    print(f'k={k}: inertia={km.inertia_:.0f}, silhouette={sil_scores[-1]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=4, color='red', linestyle='--', label='k=4 (chosen)')
axes[0].set_title('Elbow Method — Inertia vs K')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].legend()

axes[1].plot(list(K_range), sil_scores, 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=4, color='red', linestyle='--', label='k=4 (chosen)')
axes[1].set_title('Silhouette Score vs K')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('../report/elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. K-Means Clustering (k=4)

In [ ]:
km = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = km.fit_predict(rfm_scaled)

# Label clusters by mean Monetary (descending = Champions first)
cluster_means = rfm.groupby('Cluster')['Monetary'].mean().sort_values(ascending=False)
label_map = {old: new for new, old in enumerate(cluster_means.index)}
rfm['Cluster'] = rfm['Cluster'].map(label_map)

SEG_NAMES = {0: 'Champions', 1: 'Loyal', 2: 'At Risk', 3: 'Lost'}
rfm['Segment'] = rfm['Cluster'].map(SEG_NAMES)

print('Segment distribution:')
print(rfm['Segment'].value_counts())
print()
display(rfm.groupby('Segment')[['Recency','Frequency','Monetary']].mean().round(2))

## 6. Segment Profiling & Visualization

In [ ]:
SEG_ORDER  = ['Champions', 'Loyal', 'At Risk', 'Lost']
SEG_COLORS = {'Champions': '#2ecc71', 'Loyal': '#3498db',
              'At Risk': '#e67e22', 'Lost': '#e74c3c'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, feat in zip(axes, ['Recency', 'Frequency', 'Monetary']):
    data_plot = [rfm[rfm['Segment'] == s][feat].values for s in SEG_ORDER]
    bp = ax.boxplot(data_plot, patch_artist=True, labels=SEG_ORDER)
    for patch, seg in zip(bp['boxes'], SEG_ORDER):
        patch.set_facecolor(SEG_COLORS[seg])
        patch.set_alpha(0.7)
    ax.set_title(f'{feat} by Segment')
    ax.tick_params(axis='x', rotation=20)
    if feat == 'Monetary':
        ax.set_ylabel('R$')

plt.suptitle('RFM Distribution by Customer Segment', y=1.02)
plt.tight_layout()
plt.savefig('../report/rfm_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Segment count & revenue share
seg_cnt = rfm['Segment'].value_counts().reindex(SEG_ORDER)
seg_rev = rfm.groupby('Segment')['Monetary'].sum().reindex(SEG_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(seg_cnt.index, seg_cnt.values,
                   color=[SEG_COLORS[s] for s in seg_cnt.index], edgecolor='white')
for bar, val in zip(bars, seg_cnt.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}', ha='center', va='bottom', fontsize=10)
axes[0].set_title('Customers per Segment')
axes[0].set_ylabel('Count')

axes[1].pie(seg_rev.values, labels=seg_rev.index,
            colors=[SEG_COLORS[s] for s in seg_rev.index],
            autopct='%1.1f%%', startangle=140)
axes[1].set_title('Revenue Share by Segment')

plt.tight_layout()
plt.savefig('../report/segment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. PCA Visualization

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(rfm_scaled)
rfm['PC1'] = pca_coords[:, 0]
rfm['PC2'] = pca_coords[:, 1]

print(f'Explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}')

fig, ax = plt.subplots(figsize=(10, 7))
for seg in SEG_ORDER:
    sub = rfm[rfm['Segment'] == seg]
    ax.scatter(sub['PC1'], sub['PC2'], label=f'{seg} (n={len(sub):,})',
               color=SEG_COLORS[seg], alpha=0.4, s=15)
ax.set_title('K-Means Customer Clusters (PCA 2D Projection)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig('../report/pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Final segment summary
summary = rfm.groupby('Segment').agg(
    Customers   = ('customer_id', 'count'),
    Avg_Recency = ('Recency',     'mean'),
    Avg_Freq    = ('Frequency',   'mean'),
    Avg_Monetary= ('Monetary',    'mean'),
    Total_Rev   = ('Monetary',    'sum'),
).round(2).reindex(SEG_ORDER)

print('\n=== SEGMENT SUMMARY ===')
display(summary)

print('\n✅ Customer Segmentation Complete!')
print(f'Total customers segmented: {len(rfm):,}')